# Breast Cancer Histopathology Classification with Deep Learning

ResNet-18 transfer learning, evaluation metrics, confusion matrix, and model export.

> Research and educational use only; not a clinical diagnostic system.


In [ ]:
import copy
import random
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATASET_ROOT = Path(os.environ["BREAKHIS_ROOT"]).expanduser() if "BREAKHIS_ROOT" in os.environ else None
if DATASET_ROOT is None or not DATASET_ROOT.exists():
    import kagglehub
    DATASET_ROOT = Path(kagglehub.dataset_download("forderation/breakhis-400x"))
if (DATASET_ROOT / "BreaKHis 400X").exists():
    DATASET_ROOT = DATASET_ROOT / "BreaKHis 400X"
TRAIN_ROOT = DATASET_ROOT / "train"
TEST_ROOT = DATASET_ROOT / "test"
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 8
NUM_WORKERS = 0

if not TRAIN_ROOT.exists() or not TEST_ROOT.exists():
    raise FileNotFoundError(f"Expected train/test folders under {DATASET_ROOT}")
print(f"Device: {DEVICE}")

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

train_augmented = datasets.ImageFolder(TRAIN_ROOT, transform=train_transform)
train_evaluation = datasets.ImageFolder(TRAIN_ROOT, transform=eval_transform)
test_dataset = datasets.ImageFolder(TEST_ROOT, transform=eval_transform)
targets = np.asarray(train_augmented.targets)
indices = np.arange(len(train_augmented))
train_indices, val_indices = train_test_split(indices, test_size=0.2, random_state=SEED, stratify=targets)
train_dataset = Subset(train_augmented, train_indices)
val_dataset = Subset(train_evaluation, val_indices)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print("Classes:", train_augmented.classes)
print(f"Train: {len(train_dataset)}, validation: {len(val_dataset)}, test: {len(test_dataset)}")

In [ ]:
class_counts = np.bincount(targets, minlength=len(train_augmented.classes))
class_weights = len(targets) / (len(class_counts) * class_counts)
weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for parameter in model.parameters():
    parameter.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, len(train_augmented.classes))
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
print("Class counts:", dict(zip(train_augmented.classes, class_counts)))

In [ ]:
def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    correct = 0
    count = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if training:
            optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += labels.size(0)
    return total_loss / count, correct / count

history = []
best_state = None
best_val_loss = float("inf")
for epoch in range(EPOCHS):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    val_loss, val_accuracy = run_epoch(val_loader, training=False)
    history.append((train_loss, val_loss, train_accuracy, val_accuracy))
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
    print(f"Epoch {epoch + 1}/{EPOCHS} | train loss {train_loss:.4f}, acc {train_accuracy:.3f} | val loss {val_loss:.4f}, acc {val_accuracy:.3f}")

model.load_state_dict(best_state)

In [ ]:
model.eval()
all_labels, all_predictions, all_probabilities = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        probabilities = torch.softmax(model(images.to(DEVICE)), dim=1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_predictions.extend(probabilities.argmax(axis=1))
        all_probabilities.extend(probabilities[:, 1])

print(classification_report(all_labels, all_predictions, target_names=test_dataset.classes, digits=4))
print(f"ROC-AUC: {roc_auc_score(all_labels, all_probabilities):.4f}")
cm = confusion_matrix(all_labels, all_predictions)
ConfusionMatrixDisplay(cm, display_labels=test_dataset.classes).plot(cmap="Blues")
plt.title("BreaKHis test confusion matrix")
plt.show()

In [ ]:
history = np.asarray(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history[:, 0], label="train")
axes[0].plot(history[:, 1], label="validation")
axes[0].set_title("Loss")
axes[0].legend()
axes[1].plot(history[:, 2], label="train")
axes[1].plot(history[:, 3], label="validation")
axes[1].set_title("Accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()

torch.save({"model_state_dict": model.state_dict(), "classes": test_dataset.classes}, "breakhis_resnet18.pt")
print("Saved model to breakhis_resnet18.pt")